# RLT official-protocol experiments (Colab + GPU + Drive)
Monolithic notebook: model + task generators + trainer are all embedded below — no repo clone needed.
**Steps:** 1) Runtime → Change runtime type → **GPU**. 2) Set `DRIVE_FOLDER` in CONFIG and run all cells.
Start with `SMOKE = True` (minutes, validates GPU), then `SMOKE = False` for paper-scale runs.
Logs, checkpoints and summaries are written under `DRIVE_FOLDER` so they survive across sessions.


In [ ]:
# ===== CONFIG (edit me) =====
TASK = 'parity'          # parity | addition | mod5-flat | mod5-bracketed | s5-standard | s5-swaps
SPLIT = '5+3'            # encoder+decoder depth, e.g. 4+4 5+3 6+2 7+1 8+0
VARIANT = 'rlt1'         # rlt1 | rlt0 | rlt2
CHUNK_SIZE = None        # required for rlt2, e.g. 4 or 8 (B=1 recovers rlt1)
SMOKE = True             # True: tiny GPU check (~minutes). False: paper config (d=512, 2000 steps, batch 512)
DRIVE_FOLDER = 'rlt_official'   # created inside Google Drive / MyDrive /
SEEDS = (42, 43, 44)
RUN_NAME = f'{TASK}_{SPLIT}_{VARIANT}' + (f'_b{CHUNK_SIZE}' if VARIANT=='rlt2' else '') + ('_smoke' if SMOKE else '_full')
print('run:', RUN_NAME)


In [ ]:
# ===== dependencies (torch is preinstalled on Colab) =====
%pip install -q einops rotary-embedding-torch x-mlps-pytorch torch-einops-utils x-transformers tqdm fire 2>&1 | tail -2


In [ ]:
# ===== Google Drive: logs + checkpoints live here =====
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import time
DRIVE_ROOT = Path('/content/drive/MyDrive') / DRIVE_FOLDER
LOG_DIR = DRIVE_ROOT / 'logs'
CKPT_DIR = DRIVE_ROOT / 'checkpoints'
LOG_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print('drive root:', DRIVE_ROOT)
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE=='cuda' else '')


In [ ]:
# ===== RLT model (patched: alpha=0.1, RLT-0 via use_feedback=False, RLT-2 via feedback_chunk_size) =====
from __future__ import annotations

from collections import namedtuple
from functools import partial
from itertools import accumulate
from math import ceil
from typing import Callable, Sequence

import torch
from torch import nn, cat, tensor, Tensor, is_tensor
from torch.nn import Module, ModuleList, Linear, Sequential, RMSNorm, Parameter
import torch.nn.functional as F

from torch.func import functional_call

from einops import einsum, rearrange, repeat
from einops.layers.torch import Rearrange

from rotary_embedding_torch import RotaryEmbedding

from x_mlps_pytorch import MLP

from torch_einops_utils import (
    masked_mean,
    maybe,
    maybe_return,
    pack_with_inverse,
    repeat_interleave_to_match,
    temp_eval,
    tree_flatten_with_inverse,
    tree_map_detach,
    tree_map_tensor
)

# types

RLTMemories = namedtuple('RLTMemories', ['encoder_memories', 'decoder_memories'])
EncoderMemories = namedtuple('EncoderMemories', ['memories', 'keys_values'])
DecoderMemories = namedtuple('DecoderMemories', ['state', 'memories', 'recurrent_module_state'], defaults = (None,))
TransformerMemories = namedtuple('TransformerMemories', ['step', 'memories'])
Losses = namedtuple('Losses', ['cross_entropy', 'next_latent', 'kl_div'])

# constants

LinearNoBias = partial(Linear, bias = False)

# helper functions

def exists(v):
    return v is not None

def default(v, d):
    return v if exists(v) else d

def identity(t, *args, **kwargs):
    return t

def divisible_by(num, den):
    return (num % den) == 0

def uniform_like(t, low = -1., high = 1.):
    return torch.empty_like(t).uniform_(low, high)

def max_neg_value(t):
    return -torch.finfo(t.dtype).max

def to_tuple(t):
    if is_tensor(t):
        return tuple(t.tolist())
    elif isinstance(t, list):
        return tuple(t)

    return t

def slice_recurrent_lengths(recurrent_lengths, length: int, offset: int = 0):
    recurrent_lengths = to_tuple(recurrent_lengths)

    if not recurrent_lengths or length <= 0:
        return () if exists(recurrent_lengths) else None

    if isinstance(recurrent_lengths, int):
        first = min((-offset) % recurrent_lengths or recurrent_lengths, length)
        num_full, rem = divmod(length - first, recurrent_lengths)
        return (first,) + (recurrent_lengths,) * num_full + ((rem,) if rem else ())

    first, *rest = recurrent_lengths

    if offset >= first:
        return slice_recurrent_lengths(rest, length, offset - first)

    first_len = min(first - offset, length)
    return (first_len, *slice_recurrent_lengths(rest, length - first_len))

# sampling helpers

def log(t, eps = 1e-20):
    return torch.log(t.clamp(min = eps))

def gumbel_noise(t):
    noise = torch.zeros_like(t).uniform_(0, 1)
    return -log(-log(noise))

def gumbel_sample(t, temperature = 1., dim = -1, eps = 1e-10):
    if temperature == 0.:
        return t.argmax(dim = dim)

    return ((t / max(temperature, eps)) + gumbel_noise(t)).argmax(dim = dim)

# topk

def top_k(logits, frac_num_tokens = 0.1, k: int | None = None, thres: float | None = None):
    num_tokens = logits.shape[-1]

    if exists(thres):
        frac_num_tokens = 1. - thres

    k = default(k, ceil(frac_num_tokens * num_tokens))
    k = min(max(k, 1), num_tokens)

    val, ind = torch.topk(logits, k)
    probs = torch.full_like(logits, float('-inf'))
    probs.scatter_(-1, ind, val)
    return probs

# maybe flex attention

try:
    from torch.nn.attention.flex_attention import flex_attention as pt_flex_attention, create_block_mask

    if torch.cuda.is_available():
        pt_flex_attention = torch.compile(pt_flex_attention)

except ImportError:
    pt_flex_attention = None
    create_block_mask = None

def flex_attention(
    q, k, v,
    causal = False,
    sliding_window_size = None,
    scale = None
):
    assert exists(pt_flex_attention), 'flex_attention requires torch >= 2.5.0'

    rows, cols = q.shape[-2], k.shape[-2]
    single_token = rows == 1

    block_mask = None

    if (causal or exists(sliding_window_size)) and not single_token:
        prefix_len = cols - rows

        def mask_mod(b, h, q_idx, kv_idx):
            mask = True
            if causal:
                mask = mask & (q_idx + prefix_len >= kv_idx)
            if exists(sliding_window_size):
                mask = mask & (q_idx + prefix_len - kv_idx < sliding_window_size)
            return mask

        block_mask = create_block_mask(mask_mod, B = None, H = None, Q_LEN = rows, KV_LEN = cols, device = q.device)

    return pt_flex_attention(q, k, v, block_mask = block_mask, scale = scale)

# helper modules

class Scale(Module):
    def __init__(self, scale, fn: Module | None = None):
        super().__init__()
        self.scale = scale
        self.fn = default(fn, nn.Identity())

    def forward(self, *args, **kwargs):
        out = self.fn(*args, **kwargs)

        if not exists(self.scale) or self.scale == 1.:
            return out

        tensors, inverse = tree_flatten_with_inverse(out)
        first, *rest = tensors

        return inverse([first * self.scale, *rest])

# lora linear

class LoRALinear(Module):
    def __new__(
        cls,
        dim_in,
        dim_out = None,
        rank = None
    ):
        dim_out = default(dim_out, dim_in)

        if not exists(rank):
            return LinearNoBias(dim_in, dim_out)

        return super().__new__(cls)

    def __init__(
        self,
        dim_in,
        dim_out = None,
        rank = None
    ):
        super().__init__()
        dim_out = default(dim_out, dim_in)
        self.down = LinearNoBias(dim_in, rank)
        self.up = LinearNoBias(rank, dim_out)

    def forward(self, x):
        return self.up(self.down(x))

# attention

class Attention(Module):
    def __init__(
        self,
        dim,
        dim_head = 64,
        heads = 8,
        kv_heads: int | None = None,
        causal = True,
        cross_attend_key_values = False,
        rotary_embed: RotaryEmbedding | None = None,
        sliding_window_size: int | None = None,
        use_flex_attn = False
    ):
        super().__init__()
        self.scale = dim_head ** -0.5
        dim_inner = dim_head * heads
        self.causal = causal

        kv_heads = default(kv_heads, heads)
        assert divisible_by(heads, kv_heads), f'heads ({heads}) must be divisible by kv_heads ({kv_heads})'

        dim_kv_inner = dim_head * kv_heads

        self.to_queries = LinearNoBias(dim, dim_inner)
        self.to_key_values = LinearNoBias(dim, dim_kv_inner * 2) if not cross_attend_key_values else None

        self.split_queries = Rearrange('b n (h d) -> b h n d', h = heads)
        self.split_key_values = Rearrange('b n (h d) -> b h n d', h = kv_heads)
        self.merge_heads = Rearrange('b h n d -> b n (h d)')

        self.to_out = LinearNoBias(dim_inner, dim)

        self.rotary_embed = rotary_embed
        self.sliding_window_size = sliding_window_size

        assert not (use_flex_attn and not exists(pt_flex_attention)), 'flex attention is only available on torch 2.5.0 onwards'

        self.use_flex_attn = use_flex_attn

    def forward(
        self,
        tokens,
        keys_values = None,
        memories = None,
        offset = 0,
        return_memories = False,
        sliding_window_size = None
    ):
        sliding_window_size = default(sliding_window_size, self.sliding_window_size)

        q = self.to_queries(tokens)

        # keys and values can be received from the encoder

        if not exists(keys_values):
            assert exists(self.to_key_values), 'keys_values must be provided for cross attention'
            k, v = self.to_key_values(tokens).chunk(2, dim = -1)
            k, v = (self.split_key_values(t) for t in (k, v))
        else:
            k, v = keys_values

        q = self.split_queries(q)

        # rotary embedding

        if exists(self.rotary_embed):
            q = self.rotary_embed.rotate_queries_or_keys(q, offset = offset)
            k = self.rotary_embed.rotate_queries_or_keys(k, offset = offset)

        if exists(memories):
            mk, mv = memories
            k = cat((mk, k), dim = -2)
            v = cat((mv, v), dim = -2)

        return_kv = (k, v)

        # grouped query attention - repeat key / values to match the query heads, if needed

        k = repeat_interleave_to_match(k, q, dim = -3)
        v = repeat_interleave_to_match(v, q, dim = -3)

        rows, cols = q.shape[-2], k.shape[-2]
        single_token = rows == 1

        if self.use_flex_attn:
            out = flex_attention(
                q, k, v,
                causal = self.causal,
                sliding_window_size = sliding_window_size,
                scale = self.scale
            )
        else:
            sim = einsum(q, k, 'b h i d, b h j d -> b h i j') * self.scale

            mask = None
            prefix_len = cols - rows

            if self.causal and not single_token:
                causal_mask = torch.ones((rows, cols), dtype = torch.bool, device = sim.device).triu(prefix_len + 1)
                mask = causal_mask

            if exists(sliding_window_size) and not single_token:
                window_mask = torch.ones((rows, cols), dtype = torch.bool, device = sim.device).tril(prefix_len - sliding_window_size)
                mask = window_mask if mask is None else (mask | window_mask)

            if exists(mask):
                sim = sim.masked_fill(mask, max_neg_value(sim))

            attn = sim.softmax(dim = -1)

            out = einsum(attn, v, 'b h i j, b h j d -> b h i d')

        out = self.merge_heads(out)
        out = self.to_out(out)

        if not return_memories:
            return out

        return out, return_kv

# feedforward

class GEGLU(Module):
    # Shazeer et al.

    def forward(self, x):
        x, gates = x.chunk(2, dim = -1)
        return F.gelu(gates) * x

def Feedforward(
    dim,
    expansion_factor = 4.
):
    dim_inner = int(dim * expansion_factor * 2 / 3)

    return Sequential(
        Linear(dim, dim_inner * 2),
        GEGLU(),
        Linear(dim_inner, dim)
    )

# attention residual
# Guangyu (Nathan) Chen et al. with Kimi team https://arxiv.org/abs/2603.15031

class AttentionResidual(Module):
    def __init__(
        self,
        dim,
        *,
        query_key_rank: int | None = None
    ):
        super().__init__()
        self.scale = dim ** -0.5

        self.to_queries = Sequential(RMSNorm(dim), LoRALinear(dim, rank = query_key_rank))
        self.to_keys = Sequential(RMSNorm(dim), LoRALinear(dim, rank = query_key_rank))

        self.query_rmsnorm = RMSNorm(dim)
        self.key_rmsnorm = RMSNorm(dim)

    def forward(
        self,
        block_outputs: list[Tensor] | Tensor
    ):
        block_outputs = list(block_outputs)
        curr_tokens = block_outputs[-1]

        past_layers = rearrange(block_outputs, 'l b n d -> b n l d')

        queries = self.to_queries(curr_tokens)
        keys = self.to_keys(past_layers)

        queries = self.query_rmsnorm(queries)
        keys = self.key_rmsnorm(keys)

        sim = einsum(queries, keys, 'b n d, b n l d -> b n l') * self.scale

        attn = sim.softmax(dim = -1)

        return einsum(attn, past_layers, 'b n l, b n l d -> b n d')

# next-latent prediction
# Jayden Teoh et al. https://arxiv.org/abs/2511.05963

class NextLatentPrediction(Module):
    def __init__(
        self,
        dim,
        depth = 3,
        num_rollouts = 1
    ):
        super().__init__()
        self.num_rollouts = num_rollouts

        self.norm = RMSNorm(dim * 2)

        dims = (dim * 2, *(dim,) * (depth - 1), dim)

        self.mlp = MLP(*dims, activation = nn.GELU(), bias = False)

        # zero init the last linear layer so dynamics starts as identity

        nn.init.zeros_(self.mlp.layers[-1].weight)

    def rollout_loss(
        self,
        hiddens,
        token_embeds,
        teacher_logits,
        to_logits,
        labels,
        kl_loss_weight = 1.
    ):
        num_rollouts = self.num_rollouts
        assert hiddens.shape[1] > num_rollouts, f'sequence length ({hiddens.shape[1]}) must be greater than num_rollouts ({num_rollouts})'

        total_latent_loss = 0.
        total_kl_loss = 0.

        pred_h, target_h, next_tokens = hiddens, hiddens, token_embeds
        mask = labels != -1

        has_kl = kl_loss_weight > 0.
        lm_head_params = {k: v.detach() for k, v in to_logits.named_parameters()} if has_kl else None

        for i in range(num_rollouts):
            if i > 0:
                mask = mask[:, 1:]

            pred_h = pred_h[:, :-1]
            next_tokens = next_tokens[:, 1:]
            target_h = target_h[:, 1:]

            pred_h = self(pred_h, next_tokens)

            # next latent loss, stop-gradient on the target

            latent_loss = F.smooth_l1_loss(pred_h, target_h.detach(), reduction = 'none').mean(dim = -1)
            total_latent_loss = total_latent_loss + masked_mean(latent_loss, mask)

            # distill teacher next token logits through a frozen lm head

            if has_kl and teacher_logits.shape[1] > 1:
                teacher_logits = teacher_logits[:, 1:]

                student_logits = functional_call(to_logits, lm_head_params, pred_h[:, :-1])

                log_p = F.log_softmax(student_logits, dim = -1)
                log_q = F.log_softmax(teacher_logits.detach(), dim = -1)

                kl = F.kl_div(log_p, log_q, log_target = True, reduction = 'none').sum(dim = -1)
                total_kl_loss = total_kl_loss + masked_mean(kl, mask[:, 1:])

        return total_latent_loss / num_rollouts, total_kl_loss / num_rollouts

    def forward(self, current_states, next_token_embeds):
        x = cat((current_states, next_token_embeds), dim = -1)
        x = self.norm(x)
        return self.mlp(x) + current_states

# transformer

class Transformer(Module):
    def __init__(
        self,
        dim,
        *,
        depth,
        dim_head = 64,
        heads = 8,
        kv_heads: int | None = None,
        cross_attn_kv_heads: int | None = None,
        self_attn = True,
        cross_attn = False,
        self_attn_window_size = None,
        ff_expansion_factor = 4,
        rotary_embed = True,
        dim_rotary = None,
        use_flex_attn = False,
        attn_residual = False,
        attn_residual_query_key_rank: int | None = None,
        scale_residual = False
    ):
        super().__init__()
        assert not exists(self_attn_window_size) or self_attn_window_size >= 1
        assert not exists(dim_rotary) or dim_rotary <= dim_head

        self.self_attn_window_size = self_attn_window_size
        self.has_attn_residual = attn_residual

        # depth scaling for residuals (Yang et al. / Noci et al.)

        residual_scale = None

        if attn_residual:
            residual_scale = 1.
        elif isinstance(scale_residual, bool):
            residual_scale = (depth ** -0.5) if scale_residual else None
        elif isinstance(scale_residual, (int, float)):
            residual_scale = float(scale_residual)

        self.residual_scale = residual_scale

        maybe_scale_output = maybe(partial(Scale, residual_scale)) if (exists(residual_scale) and residual_scale != 1.) else identity

        # rotary embedding

        dim_rotary = default(dim_rotary, dim_head)
        self.rotary_embed = RotaryEmbedding(dim_rotary) if rotary_embed else None

        layers = ModuleList([])

        for _ in range(depth):
            self_attn_norm = RMSNorm(dim) if self_attn else None
            self_attn_module = Attention(dim = dim, dim_head = dim_head, heads = heads, kv_heads = kv_heads, rotary_embed = self.rotary_embed, sliding_window_size = self.self_attn_window_size, use_flex_attn = use_flex_attn) if self_attn else None

            cross_attn_norm = RMSNorm(dim) if cross_attn else None
            cross_attn_module = Attention(dim = dim, dim_head = dim_head, heads = heads, kv_heads = cross_attn_kv_heads, cross_attend_key_values = True, use_flex_attn = use_flex_attn) if cross_attn else None

            ff_norm = RMSNorm(dim)
            ff = Feedforward(dim = dim, expansion_factor = ff_expansion_factor)

            attn_res = AttentionResidual(dim, query_key_rank = attn_residual_query_key_rank) if attn_residual else None

            layers.append(ModuleList([
                self_attn_norm,
                maybe_scale_output(self_attn_module),
                cross_attn_norm,
                maybe_scale_output(cross_attn_module),
                ff_norm,
                maybe_scale_output(ff),
                attn_res
            ]))

        self.layers = layers

    @maybe_return('memories', 'hiddens')
    def forward(
        self,
        tokens,
        keys_values = None,
        memories = None,
        block_outputs: list[Tensor] | None = None,
        return_hiddens = False
    ):
        seq_len = tokens.shape[-2]

        step, memories = default(memories, (0, []))
        iter_memories = iter(memories)
        next_memories = []

        # keep the hidden states only if they are needed for the attention residual, or explicitly requested

        keep_hiddens = return_hiddens or self.has_attn_residual or exists(block_outputs)

        if keep_hiddens:
            block_outputs = [block_outputs] if is_tensor(block_outputs) else list(default(block_outputs, [tokens]))

        # keys and values can either be a single (keys, values) pair shared by all layers, or one pair per layer

        if exists(keys_values):
            if is_tensor(keys_values[0]):
                keys_values = (keys_values,) * len(self.layers)
            else:
                assert len(keys_values) == len(self.layers), f'expected {len(self.layers)} (keys, values) pairs, received {len(keys_values)}'

        iter_keys_values = iter(default(keys_values, ()))

        # layers

        for self_attn_norm, self_attn, cross_attn_norm, cross_attn, ff_norm, ff, attn_residual in self.layers:

            # self attention

            if exists(self_attn):
                self_attn_out, next_memory = self_attn(
                    self_attn_norm(tokens),
                    memories = next(iter_memories, None),
                    offset = step,
                    return_memories = True,
                    sliding_window_size = self.self_attn_window_size
                )

                tokens = self_attn_out + tokens

                next_memories.append(next_memory)

            # special cross attention from YOCO

            if exists(cross_attn):
                layer_keys_values = next(iter_keys_values, None)
                tokens = cross_attn(cross_attn_norm(tokens), keys_values = layer_keys_values) + tokens

            # feedforward

            tokens = ff(ff_norm(tokens)) + tokens

            # keep the hidden states for the attention residual

            if keep_hiddens:
                block_outputs.append(tokens)

            # attention residual

            if exists(attn_residual):
                tokens = attn_residual(block_outputs)

        # maybe take care of sliding window size - since always doing one token at a time, just do like inference where one slices off the earlier end

        if exists(self.self_attn_window_size):
            w = self.self_attn_window_size
            next_memories = tree_map_tensor(lambda t: t[..., -(w - 1):, :] if w > 1 else t[..., :0, :], next_memories)

        next_step = step + seq_len

        return tokens, TransformerMemories(next_step, next_memories), block_outputs

# the recurrent transition they propose

class RecurrentTransition(Module):
    def __init__(
        self,
        dim,
        alpha = 0.1,
        learned_alpha = False
    ):
        super().__init__()
        self.learned_alpha = learned_alpha
        self.alpha = Parameter(tensor(alpha)) if learned_alpha else alpha

        self.norm = RMSNorm(dim)
        self.to_gates = Linear(dim * 2, dim)
        self.to_state = LinearNoBias(dim, dim)

    def forward(
        self,
        state,
        encoded
    ):
        α = F.softplus(self.alpha) if self.learned_alpha else self.alpha

        normed_state = self.norm(state)
        encoded, normed_state = torch.broadcast_tensors(encoded, normed_state)
        gates = self.to_gates(cat((encoded, normed_state), dim = -1)).sigmoid()
        state_out = self.to_state(normed_state)

        # section 2.5, equations (2.9) - (2.11)

        return encoded + α * gates * state_out

# GLU Cross transition from Full-bandwidth Transformer (Xi Wang et al. https://arxiv.org/abs/2608.08888)

class GLUCrossTransition(Module):
    def __init__(
        self,
        dim
    ):
        super().__init__()
        self.to_state = LinearNoBias(dim, dim)
        self.to_gate = LinearNoBias(dim, dim)

    def forward(
        self,
        state,
        encoded
    ):
        encoded, state = torch.broadcast_tensors(encoded, state)
        return self.to_state(state) * self.to_gate(encoded).sigmoid()

# main class

class RLT(Module):
    def __init__(
        self,
        dim,
        *,
        enc_depth,
        dec_depth,
        num_tokens = None,
        dim_head = 64,
        heads = 8,
        kv_heads: int | None = None,
        cross_attn_kv_heads: int | None = None,
        num_kv_layers: int | None = None,
        layer_to_layer_mapping: Sequence[int] | None = None,
        ff_expansion_factor = 4.,
        recurrent_transition_alpha = 0.1,
        dec_sliding_window_size = 16,
        recurrent_block_size: int | Sequence[int] = 1,
        rotary_embed = True,
        dim_rotary = None,
        recurrent_transition: Module | None = None,
        glu_cross = False,
        use_flex_attn = False,
        tbptt_step_size: int | None = None,
        attn_residual = False,
        attn_residual_query_key_rank: int | None = None,
        attn_residual_cross_encoder = False,
        shared_weights = False,
        next_lat_loss = False,
        next_latent_loss_weight = 0.,
        next_latent_kl_loss_weight = 1.,
        next_latent_num_rollouts = 1,
        next_latent_dynamics_depth = 3,
        recurrent_state_module: Module | None = None,
        enc_depth_scale_residual = False,
        dec_depth_scale_residual = True,
        tie_embedding = True,
        jitter_noise_delta = 0.,
        use_feedback = True,
        feedback_chunk_size: int | None = None
    ):
        super().__init__()
        has_num_tokens = exists(num_tokens)
        self.has_num_tokens = has_num_tokens
        self.tie_embedding = tie_embedding
        self.recurrent_block_size = to_tuple(recurrent_block_size)

        self.token_emb = nn.Embedding(num_tokens, dim) if has_num_tokens else None

        # next latent prediction (Teoh et al. https://arxiv.org/abs/2511.05963)

        if next_lat_loss and next_latent_loss_weight == 0.:
            next_latent_loss_weight = 1.

        self.has_next_latent_loss = has_num_tokens and next_latent_loss_weight > 0.
        self.next_latent_loss_weight = next_latent_loss_weight
        self.next_latent_kl_loss_weight = next_latent_kl_loss_weight

        self.next_latent_prediction = NextLatentPrediction(
            dim = dim,
            depth = next_latent_dynamics_depth,
            num_rollouts = next_latent_num_rollouts
        ) if self.has_next_latent_loss else None

        # resolve the mapping from decoder layers to the encoded key / value layers

        if exists(layer_to_layer_mapping):
            layer_to_layer_mapping = tuple(layer_to_layer_mapping)
            num_kv_layers = default(num_kv_layers, max(layer_to_layer_mapping) + 1)

            assert len(layer_to_layer_mapping) == dec_depth, f'layer_to_layer_mapping must have length {dec_depth}'
            assert all(0 <= layer_index < num_kv_layers for layer_index in layer_to_layer_mapping), f'layer_to_layer_mapping indices must be in the range [0, {num_kv_layers})'
        else:
            num_kv_layers = default(num_kv_layers, 1)

            assert num_kv_layers in (1, dec_depth), 'layer_to_layer_mapping must be specified if num_kv_layers is neither 1 nor equal to dec_depth'

            layer_to_layer_mapping = tuple(range(dec_depth)) if num_kv_layers == dec_depth else (0,) * dec_depth

        self.layer_to_layer_mapping = layer_to_layer_mapping

        # grouped query attention - `kv_heads` applies to all attention, `cross_attn_kv_heads` overrides the cross attention

        cross_attn_kv_heads = default(cross_attn_kv_heads, default(kv_heads, heads))

        assert divisible_by(heads, cross_attn_kv_heads), f'heads ({heads}) must be divisible by cross_attn_kv_heads ({cross_attn_kv_heads})'

        dim_kv_inner = dim_head * cross_attn_kv_heads

        # whether the decoder attention residual should also attend to the encoder hiddens

        assert not attn_residual_cross_encoder or attn_residual, 'attn_residual must be enabled for the decoder attention residual to attend to encoder hiddens'

        self.attn_residual = attn_residual
        self.attn_residual_cross_encoder = attn_residual_cross_encoder

        # transformer settings

        transformer_kwargs = dict(
            heads = heads,
            kv_heads = kv_heads,
            dim_head = dim_head,
            ff_expansion_factor = ff_expansion_factor,
            rotary_embed = rotary_embed,
            dim_rotary = dim_rotary,
            use_flex_attn = use_flex_attn,
            attn_residual = attn_residual,
            attn_residual_query_key_rank = attn_residual_query_key_rank
        )

        assert not (shared_weights and enc_depth != dec_depth), f'enc_depth ({enc_depth}) must equal dec_depth ({dec_depth}) when sharing weights'

        self.shared_weights = shared_weights

        # encoder

        self.encoder = Transformer(
            dim,
            depth = enc_depth,
            scale_residual = enc_depth_scale_residual,
            **transformer_kwargs
        )

        # YOCO - Sun et al.

        self.to_encoded_key_values = Sequential(
            RMSNorm(dim),
            LinearNoBias(dim, num_kv_layers * dim_kv_inner * 2)
        )

        self.split_heads = Rearrange('b n (l h d) -> l b h n d', l = num_kv_layers, h = cross_attn_kv_heads)

        # recurrence related
        # use_feedback=False selects RLT-0 (official control): no learned
        # initial state, state norm, gated merge or feedback projection;
        # decoder receives encoder outputs directly.
        # feedback_chunk_size=B>1 selects RLT-2 chunk semantics (App. I):
        # state fixed within a chunk, committed at complete boundaries.
        # B=1 recovers RLT-1 at identical weights.

        assert use_feedback or not exists(recurrent_transition), 'RLT-0 removes the gated merge'
        assert use_feedback or not glu_cross, 'RLT-0 removes the feedback projection'
        assert use_feedback or not exists(recurrent_state_module), 'RLT-0 has no recurrent state module'
        assert use_feedback or not (jitter_noise_delta > 0.), 'RLT-0 has no recurrent state to jitter'

        if exists(feedback_chunk_size):
            assert use_feedback, 'chunked feedback requires the feedback path (RLT-2 builds on RLT-1)'
            assert isinstance(feedback_chunk_size, int) and feedback_chunk_size >= 1, 'chunk size B must be an integer >= 1'
            recurrent_block_size = feedback_chunk_size

        self.use_feedback = use_feedback
        self.feedback_chunk_size = default(feedback_chunk_size, 1) if use_feedback else None

        self.tbptt_step_size = tbptt_step_size

        if use_feedback:
            self.initial_state = nn.Parameter(torch.randn(dim) * 1e-2)

            if not exists(recurrent_transition):
                if glu_cross:
                    recurrent_transition = GLUCrossTransition(dim)
                else:
                    recurrent_transition = RecurrentTransition(dim, alpha = recurrent_transition_alpha)

            self.combine_encoded_token_and_state = recurrent_transition

            # norm the fused input from recurrence before feeding into the network (Xi Wang et al. - Full-bandwidth transformer)

            self.fused_norm = RMSNorm(dim)
        else:
            # RLT-0: direct encoder input, no feedback parameters
            self.initial_state = None
            self.combine_encoded_token_and_state = None
            self.fused_norm = nn.Identity()
            assert not exists(tbptt_step_size), 'RLT-0 uses full BPTT without a TBPTT boundary'

        self.fused_input_norm = self.fused_norm

        self.jitter_noise_delta = jitter_noise_delta
        self.has_jitter_noise_delta = jitter_noise_delta > 0.

        # recurrent state module along recurrent pathway

        self.recurrent_state_module = recurrent_state_module
        self.has_recurrent_state_module = exists(recurrent_state_module)

        # decoder

        self.decoder = Transformer(
            dim,
            depth = dec_depth,
            cross_attn = True,
            cross_attn_kv_heads = cross_attn_kv_heads,
            self_attn_window_size = dec_sliding_window_size,
            scale_residual = dec_depth_scale_residual,
            **transformer_kwargs
        )

        # maybe share weights between encoder and decoder
        # stage-specific normalizations remain separate in each Transformer, while self_attn and ff are shared

        if self.shared_weights:
            for enc_layer, dec_layer in zip(self.encoder.layers, self.decoder.layers):
                enc_layer[1] = dec_layer[1]
                enc_layer[5] = dec_layer[5]

        # to logits

        to_logits_linear = LinearNoBias(dim, num_tokens) if has_num_tokens else None

        if has_num_tokens and tie_embedding:
            to_logits_linear.weight = self.token_emb.weight

        self.to_logits = Sequential(
            RMSNorm(dim),
            to_logits_linear,
            Scale(dim ** -0.5) if tie_embedding else nn.Identity()
        ) if has_num_tokens else None

    @property
    def device(self):
        return next(self.parameters()).device

    # generate

    @temp_eval
    @torch.no_grad()
    def generate(
        self,
        prompt: Tensor,
        seq_len: int | None = None,
        max_len: int | None = None,
        temperature: float = 1.,
        filter_fn: Callable = top_k,
        filter_kwargs: dict = dict(frac_num_tokens = 0.1),
        recurrent_lengths: int | Sequence[int] | None = None
    ):
        assert self.has_num_tokens, '`num_tokens` must be passed to RLT to generate'

        recurrent_lengths = default(recurrent_lengths, self.recurrent_block_size)
        is_custom = not isinstance(recurrent_lengths, int)

        max_len = default(max_len, sum(to_tuple(recurrent_lengths)) if is_custom else seq_len)
        assert exists(max_len), 'max_len must be supplied'

        recurrent_lengths = slice_recurrent_lengths(recurrent_lengths, max_len)

        assert all(isinstance(l, int) and l > 0 for l in recurrent_lengths), 'recurrent lengths must be a sequence of positive integers'
        assert sum(recurrent_lengths) == max_len, f'sum of recurrent lengths ({sum(recurrent_lengths)}) must equal max_len ({max_len})'

        boundaries = set(accumulate(recurrent_lengths))
        is_boundary = lambda idx: idx in boundaries

        filter_fn = default(filter_fn, identity)
        filter_kwargs = default(filter_kwargs, {})

        # handle maybe 1d prompt

        prompt, inverse_pack = pack_with_inverse(prompt, '* n')

        prompt = prompt.to(self.device)

        prompt_len = prompt.shape[-1]
        sample_num_times = max(0, max_len - prompt_len)

        if sample_num_times == 0:
            return inverse_pack(prompt[:, :0])

        out = []

        # catch up memories with prompt

        prompt_recurrent_lengths = slice_recurrent_lengths(recurrent_lengths, prompt_len)

        step_out, memories = self(prompt, recurrent_lengths = prompt_recurrent_lengths, update_state = is_boundary)

        # sample first token

        logits = step_out[:, -1]
        filtered_logits = filter_fn(logits, **filter_kwargs)
        sampled = gumbel_sample(filtered_logits, temperature = temperature)
        sampled = rearrange(sampled, 'b -> b 1')

        out.append(sampled)

        # generate remaining tokens step-by-step with recurrent state

        for token_idx in range(sample_num_times - 1):
            total_index = prompt_len + 1 + token_idx

            update_state = is_boundary(total_index)

            step_out, memories = self(sampled, memories = memories, recurrent_lengths = (1,), update_state = update_state)

            logits = step_out[:, -1]
            filtered_logits = filter_fn(logits, **filter_kwargs)
            sampled = gumbel_sample(filtered_logits, temperature = temperature)
            sampled = rearrange(sampled, 'b -> b 1')

            out.append(sampled)

        # concat all newly generated tokens

        out = cat(out, dim = -1)

        return inverse_pack(out)

    def forward(
        self,
        tokens,
        memories: RLTMemories | None = None,
        return_loss = False,
        return_loss_breakdown = False,
        recurrent_lengths: int | Sequence[int] | None = None,
        update_state: bool | None = None,
        jitter_noise_delta: float | None = None
    ):
        # embed

        if self.has_num_tokens:

            if return_loss and not self.has_next_latent_loss:
                tokens, labels = tokens[:, :-1], tokens[:, 1:]

            raw_tokens = tokens
            tokens = self.token_emb(tokens)

        batch, seq_len = tokens.shape[:2]

        # memories

        enc_memories, dec_memories = default(memories, (None, None))
        enc_memories, prev_keys_values = default(enc_memories, (None, None))

        if exists(dec_memories):
            state, dec_memories, *maybe_module_state = dec_memories
            recurrent_module_state = maybe_module_state[0] if len(maybe_module_state) > 0 else None
        else:
            state, dec_memories, recurrent_module_state = None, None, None

        prev_num_tokens = prev_keys_values[0].shape[-2] if exists(prev_keys_values) else 0

        has_jitter_noise_delta = (jitter_noise_delta > 0.) if exists(jitter_noise_delta) else self.has_jitter_noise_delta
        jitter_noise_delta = default(jitter_noise_delta, self.jitter_noise_delta)

        # recurrent lengths

        block_size = default(recurrent_lengths, self.recurrent_block_size)
        is_custom = not isinstance(block_size, int)

        recurrent_lengths = to_tuple(block_size) if is_custom else slice_recurrent_lengths(block_size, seq_len, prev_num_tokens)

        assert all(isinstance(l, int) and l > 0 for l in recurrent_lengths), 'recurrent lengths must be a sequence of positive integers'
        assert sum(recurrent_lengths) == seq_len, f'sum of recurrent lengths ({sum(recurrent_lengths)}) must equal sequence length ({seq_len})'

        is_boundary = (lambda _: True) if is_custom else (lambda idx: divisible_by(idx, block_size))

        # maybe initial state (RLT-0 keeps state=None: direct encoder input)

        if self.use_feedback and not exists(state):
            state = repeat(self.initial_state, 'd -> b 1 d', b = batch)

        # encode

        encoder_out = self.encoder(
            tokens,
            memories = enc_memories,
            return_memories = True,
            return_hiddens = self.attn_residual_cross_encoder
        )

        encoded, next_enc_memories = encoder_out.tokens, encoder_out.memories
        encoder_hiddens = encoder_out.hiddens if self.attn_residual_cross_encoder else []

        # keys and values for cross attention

        keys, values = self.to_encoded_key_values(encoded).chunk(2, dim = -1)
        keys, values = (self.split_heads(t) for t in (keys, values))

        if exists(prev_keys_values):
            prev_keys, prev_values = prev_keys_values
            keys = cat((prev_keys, keys), dim = -2)
            values = cat((prev_values, values), dim = -2)

        # the main proposal, make the decoder of the YOCO setup looped

        decoder_outputs = []
        curr = 0

        for block_len in recurrent_lengths:
            encoded_block = encoded[:, curr : curr + block_len]

            # encoder hiddens, sliced to the current block, to be fed into the attention residual of the decoder

            step_encoder_hiddens = [h[:, curr : curr + block_len] for h in encoder_hiddens]

            curr += block_len
            total_index = prev_num_tokens + curr

            # maybe jitter noise along recurrent state (Xi Wang et al. - Full-bandwidth transformer)

            if self.use_feedback and self.training and has_jitter_noise_delta:
                state = state + uniform_like(state, -jitter_noise_delta, jitter_noise_delta)

            # combine encoded block with state (relies on broadcasting);
            # RLT-0 feeds encoder outputs directly (z^0_t = e_t)

            if self.use_feedback:
                decoder_block = self.combine_encoded_token_and_state(state, encoded_block)
                decoder_block = self.fused_norm(decoder_block)
            else:
                decoder_block = encoded_block

            # slice the encoded key / values to the current sequence position, and map each decoder layer to its encoded layer

            step_keys, step_values = tree_map_tensor(
                lambda t: t[list(self.layer_to_layer_mapping)],
                (keys[..., :total_index, :], values[..., :total_index, :])
            )

            step_keys_values = tuple(zip(step_keys, step_values))

            decoder_output, dec_memories = self.decoder(
                decoder_block,
                keys_values = step_keys_values,
                memories = dec_memories,
                return_memories = True,
                block_outputs = [*step_encoder_hiddens, decoder_block] if self.attn_residual else None
            )

            # append for output

            decoder_outputs.append(decoder_output)

            # set next state as decoder output only at a block boundary
            # (RLT-0: no feedback state; partial RLT-2 chunks preserve it)

            should_update_state = update_state(total_index) if callable(update_state) else default(update_state, is_boundary(total_index))

            if should_update_state and self.use_feedback:
                state = decoder_output[:, -1:]

                if self.has_recurrent_state_module:
                    module_out = self.recurrent_state_module(state, recurrent_module_state)
                    state, recurrent_module_state = module_out[:2] if isinstance(module_out, tuple) else (module_out, None)

            # truncated bptt

            if exists(self.tbptt_step_size) and divisible_by(total_index, self.tbptt_step_size):
                state, dec_memories, recurrent_module_state = tree_map_detach((state, dec_memories, recurrent_module_state))

        decoded = cat(decoder_outputs, dim = 1)

        memories = RLTMemories(
            EncoderMemories(next_enc_memories, (keys, values)),
            DecoderMemories(state, dec_memories, recurrent_module_state)
        )

        if not self.has_num_tokens:
            assert not return_loss
            return decoded, memories

        logits = self.to_logits(decoded)

        if not return_loss:
            return logits, memories

        if not self.has_next_latent_loss:
            cross_entropy_loss = F.cross_entropy(rearrange(logits, 'b n v -> b v n'), labels, ignore_index = -1)

            if not return_loss_breakdown:
                return cross_entropy_loss

            return cross_entropy_loss, Losses(cross_entropy_loss, None, None)

        # next-latent prediction loss (Teoh et al. https://arxiv.org/abs/2511.05963)

        labels = raw_tokens[:, 1:]
        teacher_logits = logits[:, :-1]

        cross_entropy_loss = F.cross_entropy(
            rearrange(teacher_logits, 'b n v -> b v n'),
            labels,
            ignore_index = -1
        )

        next_latent_loss, kl_div_loss = self.next_latent_prediction.rollout_loss(
            decoded,
            tokens,
            teacher_logits,
            self.to_logits,
            labels,
            kl_loss_weight = self.next_latent_kl_loss_weight
        )

        total_loss = (
            cross_entropy_loss +
            next_latent_loss * self.next_latent_loss_weight +
            kl_div_loss * self.next_latent_kl_loss_weight
        )

        if not return_loss_breakdown:
            return total_loss

        return total_loss, Losses(cross_entropy_loss, next_latent_loss, kl_div_loss)



In [ ]:
# ===== official task generators + metrics (App. J protocol) =====
"""Shared data generators + metrics matching the official RLT protocol.

Reference: yifanzhang-pro/recurrent-looped-tranformer, paper App. J.
  formal data seeds: train 20260914 / val 20260915 / test 20260916
  addition data seed: 42
  init seeds: 42, 43, 44 (fixed train stream + held-out across inits)
  depth-8: 2000 steps, global batch 512, micro 32
  mod-5 variants: 5000 steps

Token conventions here are explicit and minimal (the official repo ships no
code, only the spec). Loss masking follows the paper: parity / mod-5 score
the final label only; S5 scores every prefix (final-state primary);
addition scores teacher-forced answer tokens.
"""
from __future__ import annotations

import itertools
import math
import random

import torch

# seeds from App. J.4
DATA_SEED_TRAIN = 20260914
DATA_SEED_VAL = 20260915
DATA_SEED_TEST = 20260916
DATA_SEED_ADDITION = 42
INIT_SEEDS = (42, 43, 44)

# ---------------------------------------------------------------------------
# parity: binary strings 3-40 train, val {32,36,40}, test to 256
# ---------------------------------------------------------------------------

def sample_parity(rng: random.Random, length: int):
    bits = [rng.randint(0, 1) for _ in range(length)]
    return bits, sum(bits) % 2


def make_parity_batch(batch_size: int, length: int, seed: int):
    """Returns (seq [B,L] of 0/1, final parity [B]). Deterministic per seed."""
    g = torch.Generator().manual_seed(seed)
    seq = torch.randint(0, 2, (batch_size, length), generator=g)
    labels = (seq.sum(dim=-1) % 2).long()
    return seq, labels


def sample_parity_length(rng: random.Random, lo: int = 3, hi: int = 40):
    return rng.randint(lo, hi)


# ---------------------------------------------------------------------------
# addition: 1-8 digit operands, reversed-digit serialization, teacher-forced
# vocab: 0-9 digits, 10='+', 11='=', 12=BOS, 13=EOS, 14=PAD
# format: BOS rev(a) + rev(b) = rev(a+b) EOS   (all digit fields reversed)
# ---------------------------------------------------------------------------

ADD_VOCAB = 15
ADD_BOS, ADD_EOS, ADD_PAD = 12, 13, 14
ADD_PLUS, ADD_EQ = 10, 11


def sample_addition(rng: random.Random, lo_digits: int = 1, hi_digits: int = 8):
    da = rng.randint(lo_digits, hi_digits)
    db = rng.randint(lo_digits, hi_digits)
    a = rng.randint(0 if da == 1 else 10 ** (da - 1), 10**da - 1)
    b = rng.randint(0 if db == 1 else 10 ** (db - 1), 10**db - 1)
    return a, b


def encode_addition(a: int, b: int, max_len: int | None = None):
    ra = list(map(int, str(a)[::-1]))
    rb = list(map(int, str(b)[::-1]))
    rc = list(map(int, str(a + b)[::-1]))
    toks = [ADD_BOS] + ra + [ADD_PLUS] + rb + [ADD_EQ] + rc + [ADD_EOS]
    # mask: 1 on answer tokens (rev sum + EOS), 0 elsewhere/pad
    ans_start = 1 + len(ra) + 1 + len(rb) + 1
    mask = [0] * ans_start + [1] * (len(rc) + 1)
    if max_len is not None:
        pad = max_len - len(toks)
        toks += [ADD_PAD] * pad
        mask += [0] * pad
    return torch.tensor(toks, dtype=torch.long), torch.tensor(mask, dtype=torch.bool)


def addition_max_len(max_digits: int = 8):
    # BOS + 8 + PLUS + 8 + EQ + 9 (carry) + EOS
    return 1 + max_digits + 1 + max_digits + 1 + (max_digits + 1) + 1


# ---------------------------------------------------------------------------
# mod-5: flat (no brackets, odd lens 3-39) and bracketed trees (3-40)
# vocab: 0-4 digits, 5='+', 6='-', 7='*', 8='(', 9=')', 10='=', 11=BOS,
#        12=EOS, 13=PAD. Final value (0-4) supervised.
# ---------------------------------------------------------------------------

MOD_VOCAB = 14
MOD_BOS, MOD_EOS, MOD_PAD = 11, 12, 13
MOD_OP = {"+": 5, "-": 6, "*": 7}
MOD_LP, MOD_RP, MOD_EQ = 8, 9, 10


def eval_flat_mod5(nums: list[int], ops: list[str]) -> int:
    # * before +,-  (standard precedence, left-assoc)
    vals, cur_ops = [nums[0]], []
    for op, n in zip(ops, nums[1:]):
        if op == "*":
            vals[-1] = (vals[-1] * n) % 5
        else:
            vals.append(n)
            cur_ops.append(op)
    total = vals[0]
    for op, v in zip(cur_ops, vals[1:]):
        total = (total + v) % 5 if op == "+" else (total - v) % 5
    return total % 5


def sample_flat_mod5(rng: random.Random, odd_len: int):
    n_nums = (odd_len + 1) // 2
    nums = [rng.randint(0, 4) for _ in range(n_nums)]
    ops = [rng.choice(["+", "-", "*"]) for _ in range(n_nums - 1)]
    return nums, ops, eval_flat_mod5(nums, ops)


def encode_flat_mod5(nums, ops):
    toks = [MOD_BOS]
    for i, n in enumerate(nums):
        toks.append(n)
        if i < len(ops):
            toks.append(MOD_OP[ops[i]])
    toks += [MOD_EQ]
    return torch.tensor(toks, dtype=torch.long), eval_flat_mod5(nums, ops)


def sample_bracketed_mod5(rng: random.Random, target_len: int):
    """Random full binary tree with +-* internal nodes, leaves 0-4.

    target_len counts payload tokens (digits+ops+parens, excl. BOS/EQ).
    Grows a tree until the payload reaches/exceeds target, then trims by
    collapsing back — simple approach: build random tree with bounded leaves.
    """
    n_leaves = max(2, (target_len + 1) // 4)
    return _random_tree(rng, n_leaves)


def _random_tree(rng: random.Random, n_leaves: int):
    if n_leaves == 1:
        v = rng.randint(0, 4)
        return ([str(v)], v)
    left_n = rng.randint(1, n_leaves - 1)
    lt, lv = _random_tree(rng, left_n)
    rt, rv = _random_tree(rng, n_leaves - left_n)
    op = rng.choice(["+", "-", "*"])
    if op == "+":
        v = (lv + rv) % 5
    elif op == "-":
        v = (lv - rv) % 5
    else:
        v = (lv * rv) % 5
    return (["("] + lt + [op] + rt + [")"], v)


def encode_bracketed_mod5(tree_toks: list[str]):
    mp = {"+": 5, "-": 6, "*": 7, "(": 8, ")": 9}
    toks = [MOD_BOS] + [mp.get(t, int(t)) for t in tree_toks] + [MOD_EQ]
    return torch.tensor(toks, dtype=torch.long)


# ---------------------------------------------------------------------------
# S5: symmetric group on 5 elements, 120 perms.
# standard: all 120; swaps: identity + 10 single transpositions.
# compose 32 perms in training; supervise running product after each input.
# Represent perm as tuple of 5 images of 0..4. Product: apply left then right:
#   (p*q)[i] = q[p[i]]  (running product p_{k} = p_{k-1} composed with input)
# ---------------------------------------------------------------------------

def _all_s5():
    return list(itertools.permutations(range(5)))


ALL_S5 = _all_s5()
IDENTITY = (0, 1, 2, 3, 4)


def _transpositions():
    out = [IDENTITY]
    for i in range(5):
        for j in range(i + 1, 5):
            p = list(range(5))
            p[i], p[j] = p[j], p[i]
            out.append(tuple(p))
    return out


SWAPS_S5 = _transpositions()  # 1 + 10 = 11
PERM_TO_IDX = {p: i for i, p in enumerate(ALL_S5)}


def compose(p, q):
    return tuple(q[p[i]] for i in range(5))


def sample_s5(rng: random.Random, length: int, variant: str):
    pool = ALL_S5 if variant == "standard" else SWAPS_S5
    inputs = [rng.choice(pool) for _ in range(length)]
    states, cur = [], IDENTITY
    for x in inputs:
        cur = compose(cur, x)
        states.append(PERM_TO_IDX[cur])
    return [PERM_TO_IDX[x] for x in inputs], states


def s5_batch(batch_size: int, length: int, variant: str, seed: int):
    g = torch.Generator().manual_seed(seed)
    pool_n = 120 if variant == "standard" else 11
    idx = torch.randint(0, pool_n, (batch_size, length), generator=g)
    pool = ALL_S5 if variant == "standard" else SWAPS_S5
    inputs = torch.tensor([[PERM_TO_IDX[pool[i]] for i in row] for row in idx.tolist()])
    # running products
    states = torch.empty_like(inputs)
    for b in range(batch_size):
        cur = IDENTITY
        for t in range(length):
            cur = compose(cur, pool[idx[b, t].item()])
            states[b, t] = PERM_TO_IDX[cur]
    return inputs, states


# ---------------------------------------------------------------------------
# metrics: mean +- sample SD (ddof=1) over init seeds, matching App. J.3
# ---------------------------------------------------------------------------

def mean_sample_sd(vals: list[float]):
    n = len(vals)
    m = sum(vals) / n
    if n < 2:
        return m, 0.0
    var = sum((v - m) ** 2 for v in vals) / (n - 1)
    return m, math.sqrt(var)



In [ ]:

# ===== trainer: official hyperparams, everything logged to Drive =====
import json, math, random, time
import torch
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from x_transformers import TransformerWrapper, Decoder
from tqdm.auto import tqdm  # notebook widget bar (ETA/rate) in Colab, terminal bar in scripts

BASE_LR, MIN_LR, WARMUP = 1e-4, 5e-6, 200
def lr_schedule(step, total):
    if step < WARMUP: return BASE_LR * (step + 1) / WARMUP
    p = (step - WARMUP) / max(1, total - WARMUP)
    return MIN_LR + 0.5 * (BASE_LR - MIN_LR) * (1 + math.cos(math.pi * p))

def build(dim, heads, enc, dec, vocab, swa=8, variant='rlt1', chunk_size=None):
    dim_head = dim // heads
    ffn_mult = 1365 / 512
    plain = TransformerWrapper(num_tokens=vocab, max_seq_len=2048,
        attn_layers=Decoder(dim=dim, depth=enc+dec, heads=heads, attn_dim_head=dim_head,
            rotary_pos_emb=True, rotary_emb_dim=min(dim_head, 64),
            use_rmsnorm=True, ff_glu=True, ff_mult=ffn_mult)).to(DEVICE)
    kw = dict(num_tokens=vocab, dim=dim, enc_depth=enc, dec_depth=dec, heads=heads,
              dim_head=dim_head, dec_sliding_window_size=swa, num_kv_layers=1,
              ff_expansion_factor=ffn_mult)
    if variant == 'rlt0':
        rlt = RLT(use_feedback=False, **kw).to(DEVICE)
    elif variant == 'rlt2':
        assert chunk_size and chunk_size >= 1
        rlt = RLT(recurrent_transition_alpha=0.1, tbptt_step_size=128,
                  feedback_chunk_size=chunk_size, **kw).to(DEVICE)
    else:
        rlt = RLT(recurrent_transition_alpha=0.1, tbptt_step_size=128, **kw).to(DEVICE)
    return plain, rlt

def vocab_of(task):
    return {'parity': 2, 'addition': 15, 'mod5-flat': 14,
            'mod5-bracketed': 14, 's5-standard': 120, 's5-swaps': 120}[task]
def dv(t): return t.to(DEVICE) if isinstance(t, torch.Tensor) else t

def sample_train_batch(task, micro, rng):
    if task == 'parity':
        L = rng.randint(3, 40)
        s, l = make_parity_batch(micro, L, rng.randint(0, 2**31 - 1))
        return ('parity', dv(s), dv(l), L)
    if task == 'addition':
        Lm = addition_max_len()
        toks, masks = [], []
        for _ in range(micro):
            da, db = rng.randint(1, 8), rng.randint(1, 8)
            a = rng.randint(0 if da == 1 else 10**(da-1), 10**da - 1)
            b = rng.randint(0 if db == 1 else 10**(db-1), 10**db - 1)
            t, m = encode_addition(a, b, max_len=Lm)
            toks.append(t); masks.append(m)
        return ('addition', dv(torch.stack(toks)), None, dv(torch.stack(masks)))
    if task in ('mod5-flat', 'mod5-bracketed'):
        seqs, labs = [], []
        for _ in range(micro):
            if task == 'mod5-flat':
                L = rng.choice([x for x in range(3, 40, 2)])
                nums = [rng.randint(0, 4) for _ in range((L + 1) // 2)]
                ops = [rng.choice(['+', '-', '*']) for _ in range((L - 1) // 2)]
                t = encode_flat_mod5(nums, ops)[0]; labs.append(eval_flat_mod5(nums, ops))
            else:
                tt, v = sample_bracketed_mod5(rng, rng.randint(3, 40))
                t = encode_bracketed_mod5(tt); labs.append(v)
            seqs.append(t)
        pad = torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=MOD_PAD)
        return ('mod5', dv(pad), dv(torch.tensor(labs)), None)
    v = 'standard' if task == 's5-standard' else 'swaps'
    s, st = s5_batch(micro, 32, v, rng.randint(0, 2**31 - 1))
    return ('s5', dv(s), dv(st), 32)

def batch_loss(model, task, batch):
    get = lambda o: o[0] if isinstance(o, tuple) else o
    kind = batch[0]
    if kind == 'parity':
        _, seq, lab, _ = batch
        return F.cross_entropy(get(model(seq))[:, -1], lab)
    if kind == 'addition':
        _, seq, _, mask = batch
        logits = get(model(seq[:, :-1]))
        tgt = seq[:, 1:].clone(); tgt[~mask[:, 1:]] = -100
        return F.cross_entropy(logits.transpose(1, 2), tgt, ignore_index=-100)
    if kind == 'mod5':
        _, seq, lab, _ = batch
        return F.cross_entropy(get(model(seq))[:, -1], lab)
    _, seq, states, _ = batch
    return F.cross_entropy(get(model(seq)).transpose(1, 2), states)

def test_lengths(task):
    if task == 'parity': return [32, 40, 48, 64, 96, 128, 192, 256]
    if task == 'addition': return [9, 10, 11, 12, 14, 16, 32]
    if task == 'mod5-flat': return [31, 39, 47, 63, 95, 127, 191, 255]
    if task == 'mod5-bracketed': return [32, 40, 48, 64, 96, 128, 192, 256]
    return [32, 48, 64, 96, 128, 192, 256, 512]

@torch.no_grad()
def test_acc(model, task, length):
    get = lambda o: o[0] if isinstance(o, tuple) else o
    model.eval()
    if task == 'parity':
        s, l = make_parity_batch(1024, length, DATA_SEED_TEST + length)
        return (get(model(dv(s)))[:, -1].argmax(-1).cpu() == l).float().mean().item() * 100
    if task in ('s5-standard', 's5-swaps'):
        v = 'standard' if task == 's5-standard' else 'swaps'
        s, st = s5_batch(1024, length, v, DATA_SEED_TEST + length)
        return (get(model(dv(s))).argmax(-1).cpu()[:, -1] == st[:, -1]).float().mean().item() * 100
    return float('nan')

def log_jsonl(path, rec):
    with open(path, 'a') as f: f.write(json.dumps(rec) + '\n')

# New tasks (chain-mod5, bracketed-steps, parity-noise): see docs/new-tasks-protocol.md
def run(task, split, total_steps, global_batch, micro_batch, dim, heads, init_seed, variant, chunk_size):
    torch.manual_seed(init_seed); random.seed(init_seed)
    enc, dec = map(int, split.split('+'))
    tag = variant if variant != 'rlt2' else f'rlt2-b{chunk_size}'
    out = {}
    for name, model in zip((f'plain-{enc+dec}', f'{tag}-{split}'),
                           build(dim, heads, enc, dec, vocab_of(task), variant=variant, chunk_size=chunk_size)):
        opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, betas=(0.9, 0.95), weight_decay=0.1)
        rng = random.Random(DATA_SEED_TRAIN + init_seed)
        best, best_state, step_times = float('inf'), None, []
        accum = max(1, global_batch // micro_batch)
        pbar = tqdm(range(total_steps), desc=f'{task}/{name}/s{init_seed}', leave=False)
        for step in pbar:
            t0 = time.perf_counter()
            model.train(); opt.zero_grad()
            tot = 0.0
            for _ in range(accum):
                l = batch_loss(model, task, sample_train_batch(task, micro_batch, rng)) / accum
                l.backward()
                tot += l.item()
            for g in opt.param_groups: g['lr'] = lr_schedule(step, total_steps)
            clip_grad_norm_(model.parameters(), 1.0); opt.step()
            dt = time.perf_counter() - t0
            step_times.append(dt)
            pbar.set_postfix(loss=f'{tot:.3f}', s_it=f'{dt:.1f}')
            if (step + 1) % 100 == 0 or step == total_steps - 1:
                vl = batch_loss(model, task, sample_train_batch(task, 64, random.Random(DATA_SEED_VAL))).item()
                pbar.set_postfix(val=f'{vl:.3f}')
                log_jsonl(str(LOG_DIR / f'{RUN_NAME}.jsonl'),
                          {'type': 'val', 'model': name, 'seed': init_seed, 'step': step + 1, 'val_loss': vl,
                           'step_s': sum(step_times[-100:]) / min(100, len(step_times))})
                if vl < best:
                    best = vl
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
        gen = {L: test_acc(model, task, L) for L in test_lengths(task)}
        timed = step_times[1000:2000] if total_steps > 1000 else step_times
        out[name] = {'best_val': best, 'generalization': gen, 'mean_step_s': sum(timed) / max(1, len(timed))}
        torch.save(best_state, CKPT_DIR / f'{RUN_NAME}-{name}-seed{init_seed}.pt')
    return out

# paper config, or tiny smoke to validate the GPU session
if SMOKE: total_steps, global_batch, micro_batch, dim, heads = 20, 32, 8, 64, 4
else:
    total_steps, global_batch, micro_batch, dim, heads = 2000, 512, 32, 512, 4
    if TASK.startswith('mod5'): total_steps = 5000
log_jsonl(str(LOG_DIR / f'{RUN_NAME}.jsonl'), {'type': 'run_config', 'task': TASK, 'split': SPLIT,
    'variant': VARIANT, 'chunk': CHUNK_SIZE, 'steps': total_steps, 'batch': global_batch,
    'dim': dim, 'device': DEVICE, 'seeds': list(SEEDS)})
per_seed = [run(TASK, SPLIT, total_steps, global_batch, micro_batch, dim, heads, s, VARIANT, CHUNK_SIZE) for s in SEEDS]
summary = {}
for n in per_seed[0]:
    summary[n] = {}
    for L in test_lengths(TASK):
        vals = [r[n]['generalization'][L] for r in per_seed]
        m, sd = mean_sample_sd([v for v in vals if v == v])
        summary[n][L] = {'mean': m, 'sd': sd}
steps = {n: sum(r[n]['mean_step_s'] for r in per_seed) / len(per_seed) for n in per_seed[0]}
Path = __import__('pathlib').Path
Path(LOG_DIR / f'{RUN_NAME}_summary.json').write_text(json.dumps(
    {'run': RUN_NAME, 'summary': {k: {str(k2): v2 for k2, v2 in v.items()} for k, v in summary.items()},
     'mean_step_s': steps}, indent=2))
print(f'\n{TASK} {SPLIT} {VARIANT} (mean +- SD over {SEEDS}):')
for L in test_lengths(TASK):
    print(f"len {L:3d}: " + '  '.join(f'{n} {summary[n][L]["mean"]:5.1f}+-{summary[n][L]["sd"]:4.1f}' for n in summary))
print('mean step s:', '  '.join(f'{n} {steps[n]:.2f}' for n in steps))
print('saved to', DRIVE_ROOT)





In [ ]:
# ===== collect every run stored in Drive =====
import json, glob
from pathlib import Path
for p in sorted(glob.glob(str(LOG_DIR / '*_summary.json'))):
    d = json.loads(Path(p).read_text())
    print('---', d['run'], '| step-s:', {k: round(v,2) for k,v in d['mean_step_s'].items()})
    for L, row in list(d['summary'][list(d['summary'])[-1]].items())[:3]:
        print('   len', L, {m: f"{v['mean']:.1f}+-{v['sd']:.1f}" for m, v in
              ((m, d['summary'][m][L]) for m in d['summary'])})


## Interpreting results
Smoke runs sit at chance (parity ~50%, s5 ~0.8%) — they only prove the GPU pipeline works.
Full runs: expect parity 5+3/7+1 → 100% at 256 bits in all seeds vs Transformer ~50%; swaps-S5 4+4 → ~56% final-state at 512 ops vs ~0.9%.
Compare your `*_summary.json` tables against paper Tables 1–2 (Sept 17 snapshot). Step-time ordering should read RLT-1 slowest, RLT-0 fastest.
